# Milestone 4
Synthetic mashup generation, mel feature extraction, and a CRNN baseline.

In [2]:
import glob
import os
import random
from pathlib import Path

import numpy as np
import torch
import torchaudio

INPUT_BASE = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup"
WORKING_BASE = "/kaggle/working"

STEMS_PATH = os.path.join(INPUT_BASE, "genres_stems")
NOISE_PATH = os.path.join(INPUT_BASE, "ESC-50-master/audio")
OUTPUT_PATH = os.path.join(WORKING_BASE, "synthetic_mashups/train")


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


seed_everything(42)


def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    genres = [
        "blues", "classical", "country", "disco", "hiphop",
        "jazz", "metal", "pop", "reggae", "rock",
    ]
    target_length = target_sr * duration
    noise_files = glob.glob(os.path.join(noise_dir, "**", "*.wav"), recursive=True)

    for genre in genres:
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)

        song_folders = glob.glob(os.path.join(stems_dir, genre, "*"))
        if not song_folders:
            print(f"Warning: no songs found for {genre}")
            continue

        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ["drums.wav", "vocals.wav", "bass.wav", "other.wav"]

            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if not os.path.exists(stem_path):
                    continue

                waveform, sr = torchaudio.load(stem_path)
                if sr != target_sr:
                    resampler = torchaudio.transforms.Resample(sr, target_sr)
                    waveform = resampler(waveform)

                if waveform.shape[1] > target_length:
                    waveform = waveform[:, :target_length]
                elif waveform.shape[1] < target_length:
                    pad = target_length - waveform.shape[1]
                    waveform = torch.nn.functional.pad(waveform, (0, pad))

                stems.append(waveform)

            if len(stems) != 4:
                continue

            mashup = torch.stack(stems).sum(dim=0)
            mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)

            noise_file = random.choice(noise_files)
            noise, _ = torchaudio.load(noise_file)
            if noise.shape[1] > target_length:
                noise = noise[:, :target_length]

            start_idx = random.randint(0, target_length - noise.shape[1])
            intensity = random.uniform(0.1, 0.4)
            mashup[:, start_idx:start_idx + noise.shape[1]] += noise * intensity
            mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)

            out_path = genre_out_dir / f"mashup_{i:03d}.wav"
            torchaudio.save(str(out_path), mashup, target_sr)


generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)


def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr,
        n_fft=2048,
        hop_length=512,
        n_mels=128,
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
    wav_files = glob.glob(os.path.join(input_dir, "**", "*.wav"), recursive=True)

    if not wav_files:
        print(f"Warning: no .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = (Path(output_dir) / rel_path).with_suffix(".pt")
        out_path.parent.mkdir(parents=True, exist_ok=True)

        waveform, sr = torchaudio.load(wav_path)
        if sr != target_sr:
            resampler = torchaudio.transforms.Resample(sr, target_sr)
            waveform = resampler(waveform)

        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        torch.save(mel_spec_db, out_path)

    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


INPUT_DIR = "/kaggle/working/synthetic_mashups/train"
OUTPUT_DIR = "/kaggle/working/features/train"

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)


Successfully saved 500 feature files to /kaggle/working/features/train


## Quick Checks

In [3]:
from pathlib import Path

import torchaudio

root = Path("/kaggle/working/synthetic_mashups/train")
wav_path = next(root.rglob("*.wav"))

waveform, sr = torchaudio.load(str(wav_path))

print("file:", wav_path)
print("sample_rate:", sr)
print("shape_tuple:", tuple(waveform.shape))
print("expected_num_samples:", 22050 * 30)
print("actual_num_samples:", waveform.shape[1])


file: /kaggle/working/synthetic_mashups/train/classical/mashup_018.wav
sample_rate: 22050
shape_tuple: (2, 661500)
expected_num_samples: 661500
actual_num_samples: 661500


In [5]:
from pathlib import Path

import torch

candidate_dirs = [
    Path("/kaggle/working/features/train"),
    Path("/kaggle/working/synthetic_mashups_features/train"),
    Path("/kaggle/working/synthetic_mashups/train_features"),
    Path("/kaggle/working/synthetic_mashups/features/train"),
    Path("/kaggle/working"),
]

pt_path = None
for directory in candidate_dirs:
    if not directory.exists():
        continue
    pt_path = next(directory.rglob("*.pt"), None)
    if pt_path is not None:
        break

assert pt_path is not None, "No .pt feature file found."

obj = torch.load(pt_path, map_location="cpu")

if torch.is_tensor(obj):
    shape = tuple(obj.shape)
elif isinstance(obj, dict):
    tensor_items = {k: v for k, v in obj.items() if torch.is_tensor(v)}
    assert tensor_items, f"No tensor found in dict keys: {list(obj.keys())}"
    first_key = next(iter(tensor_items))
    shape = tuple(tensor_items[first_key].shape)
else:
    raise TypeError(f"Unsupported object type in {pt_path}: {type(obj)}")

print("file:", pt_path)
print("shape:", shape)


file: /kaggle/working/features/train/classical/mashup_027.pt
shape: (2, 128, 1292)


## CRNN Baseline

In [6]:
import torch
import torch.nn as nn


class CRNN(nn.Module):
    def __init__(self, num_classes=10, n_mels=128):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        lstm_input_size = 64 * (n_mels // 4)
        self.rnn = nn.LSTM(
            input_size=lstm_input_size,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.fc = nn.Linear(64 * 2, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = x.permute(0, 3, 1, 2).contiguous()
        b, t, c, f = x.shape
        x = x.view(b, t, c * f)
        x, _ = self.rnn(x)
        x = torch.max(x, dim=1).values
        return self.fc(x)


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CRNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_epochs = 10
